# Confidence Filtering v5: Class Prototype Distance

## 핵심 아이디어

confidence = **label accuracy** (이 오디오가 부여된 class label에 얼마나 정확하게 해당하는가)

v3/v4의 한계: audio embedding과 class label을 **별개 feature**로 입력  
v5의 개선: 이 둘의 **관계** — audio/text embedding이 assigned class와 얼마나 잘 맞는지 — 를 직접 feature화

## 신규 Feature: Class Prototype Distance (6개)

| Feature | 의미 |
|---------|------|
| `audio_class_sim` | audio embedding ↔ assigned class 오디오 centroid cosine 유사도 |
| `text_class_sim` | text embedding ↔ assigned class 텍스트 centroid cosine 유사도 |
| `audio_top_sim` | audio embedding ↔ top-class 오디오 centroid cosine 유사도 |
| `text_top_sim` | text embedding ↔ top-class 텍스트 centroid cosine 유사도 |
| `audio_margin` | audio_class_sim - best_other_class_sim (명확한 라벨일수록 큰 값) |
| `text_margin` | text_class_sim - best_other_class_sim |

**Centroid는 학습 fold의 confidence >= 4 샘플만 사용** → data leakage 없음

## 3가지 실험

| 실험 | Feature | Target | 핵심 질문 |
|------|---------|--------|----------|
| **v5-A** | baseline + prototype (6d) | binary (conf>=4) | prototype feature가 성능을 올리는가? |
| **v5-B** | audio+text+class+top_class + prototype (metadata 제거) | binary | metadata numeric은 noise인가? |
| **v5-C** | baseline + prototype (6d) | ordinal (4 thresholds) | binary 대신 ordinal target이 더 좋은가? |

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

try:
    display
except NameError:
    def display(obj): print(obj)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import confidence_model_v2_experiments as v2

OUTPUT_DIR = ROOT / "outputs" / "confidence_filter_v5"
REPORT_DIR = OUTPUT_DIR / "reports"
PRED_DIR   = OUTPUT_DIR / "predictions"
PLOT_DIR   = OUTPUT_DIR / "plots"
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
for d in [REPORT_DIR, PRED_DIR, PLOT_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
DENSE_THRESHOLDS = np.round(np.linspace(0.01, 0.99, 197), 4)

RUN_CONFIG = {
    "seed":          SEED,
    "folds":         5,
    "epochs":        60,
    "patience":      10,
    "batch_size":    256,
    "learning_rate": 1e-3,
    "weight_decay":  1e-4,
    "dropout":       0.3,
    "hidden":        [512, 256],
}

print("device:", DEVICE)
print("root:  ", ROOT)

In [ ]:
# ── Load BSD10k data ──────────────────────────────────────────────────────────
v2.seed_everything(SEED)
parts, class_categories, top_class_categories = v2.load_bsd10k_parts()
bsd10k_df = parts["df"].copy()
y_conf   = bsd10k_df["confidence"].to_numpy(dtype=np.int64)
y_binary = (y_conf >= 4).astype(np.int64)

print(f"BSD10k samples with embeddings : {len(bsd10k_df):,}")
print(f"high confidence (>=4) ratio    : {y_binary.mean():.4f}")
print(f"class categories               : {len(class_categories)}")
print(f"top class categories           : {len(top_class_categories)}")
display(pd.Series(y_conf).value_counts().sort_index().rename_axis("confidence").reset_index(name="n"))

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark    = False
    torch.backends.cudnn.deterministic = True


def safe_auc_pr(y_true, score):
    try:
        return float(average_precision_score(y_true, score))
    except ValueError:
        return float("nan")


def metrics_at_threshold(y_true, score, threshold):
    pred = (score >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy" : float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall"   : float(recall_score(y_true, pred, zero_division=0)),
        "f1"       : float(f1_score(y_true, pred, zero_division=0)),
        "auc_pr"   : safe_auc_pr(y_true, score),
    }


def threshold_sweep(y_true, score, thresholds=DENSE_THRESHOLDS):
    rows = []
    for t in thresholds:
        m = metrics_at_threshold(y_true, score, t)
        rows.append({k: m[k] for k in ["threshold", "accuracy", "precision", "recall", "f1"]})
    return pd.DataFrame(rows)


def pick_f1_optimal(y_true, score, thresholds=DENSE_THRESHOLDS):
    sweep = threshold_sweep(y_true, score, thresholds)
    best  = sweep.sort_values(["f1", "precision", "threshold"], ascending=[False, False, True]).iloc[0]
    return best, sweep


def markdown_table(df, float_digits=4):
    view = df.copy()
    for col in view.columns:
        if pd.api.types.is_float_dtype(view[col]):
            view[col] = view[col].map(lambda x: "" if pd.isna(x) else f"{x:.{float_digits}f}")
    view    = view.fillna("")
    headers = [str(c) for c in view.columns]
    rows    = [[str(v) for v in row] for row in view.to_numpy()]
    widths  = [max(len(headers[i]), max((len(r[i]) for r in rows), default=0))
               for i in range(len(headers))]
    header  = "| " + " | ".join(headers[i].ljust(widths[i]) for i in range(len(headers))) + " |"
    sep     = "| " + " | ".join("-" * widths[i]            for i in range(len(headers))) + " |"
    body    = ["| " + " | ".join(r[i].ljust(widths[i])     for i in range(len(headers))) + " |"
               for r in rows]
    return "\n".join([header, sep] + body)


def percentile_rank(x):
    return pd.Series(x).rank(method="average", pct=True).to_numpy(dtype=np.float32)


print("helpers loaded.")

In [ ]:
# ── Prototype feature computation (fold-safe) ─────────────────────────────────
#
# confidence = label accuracy → 핵심 signal:
#   이 audio/text embedding이 자신의 assigned class와 얼마나 잘 맞는가?
#
# 중요: centroid_mask로 학습 fold의 high-confidence 샘플만 사용
#       → validation fold가 centroid에 기여하지 않아 data leakage 없음

def normalize_rows(x):
    norm = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norm, 1e-8)


def build_centroids(emb_norm, labels, categories, mask):
    """
    mask: boolean (n,) — centroid 계산에 사용할 샘플 집합
    category별 L2-normalized centroid를 반환한다.
    mask 내 해당 category 샘플이 없으면 전체 샘플로 fallback.
    """
    centroids = {}
    for cat in categories:
        m = (labels == cat) & mask
        if m.sum() == 0:
            m = (labels == cat)                       # fallback
        proto = emb_norm[m].mean(axis=0) if m.sum() > 0 else np.zeros(emb_norm.shape[1])
        n_    = np.linalg.norm(proto)
        centroids[cat] = (proto / max(n_, 1e-8)).astype(np.float32)
    return centroids


def compute_prototype_features(df, audio_norm, text_norm,
                                class_categories, top_class_categories,
                                centroid_mask):
    """
    Returns (n, 6) array:
      [audio_class_sim, text_class_sim, audio_top_sim, text_top_sim,
       audio_margin, text_margin]
    """
    cls_arr = df["class"].astype(str).to_numpy()
    top_arr = df["class_top"].astype(str).to_numpy()
    n       = len(df)

    audio_ctr = build_centroids(audio_norm, cls_arr, class_categories,     centroid_mask)
    text_ctr  = build_centroids(text_norm,  cls_arr, class_categories,     centroid_mask)
    audio_top = build_centroids(audio_norm, top_arr, top_class_categories, centroid_mask)
    text_top  = build_centroids(text_norm,  top_arr, top_class_categories, centroid_mask)

    A = np.vstack([audio_ctr[c] for c in class_categories])   # (n_cls, 512)
    T = np.vstack([text_ctr[c]  for c in class_categories])
    cls_to_idx = {c: i for i, c in enumerate(class_categories)}

    all_audio_sims = audio_norm @ A.T    # (n, n_cls)
    all_text_sims  = text_norm  @ T.T

    audio_assigned = np.zeros(n, dtype=np.float32)
    text_assigned  = np.zeros(n, dtype=np.float32)
    audio_top_sim  = np.zeros(n, dtype=np.float32)
    text_top_sim   = np.zeros(n, dtype=np.float32)
    audio_margin   = np.zeros(n, dtype=np.float32)
    text_margin    = np.zeros(n, dtype=np.float32)

    for i in range(n):
        cls = cls_arr[i]
        top = top_arr[i]
        j   = cls_to_idx.get(cls, 0)

        audio_assigned[i] = all_audio_sims[i, j]
        text_assigned[i]  = all_text_sims[i, j]

        others_a = np.delete(all_audio_sims[i], j)
        others_t = np.delete(all_text_sims[i],  j)
        audio_margin[i] = audio_assigned[i] - (others_a.max() if len(others_a) > 0 else 0.0)
        text_margin[i]  = text_assigned[i]  - (others_t.max() if len(others_t) > 0 else 0.0)

        audio_top_sim[i] = float(audio_norm[i] @ audio_top[top])
        text_top_sim[i]  = float(text_norm[i]  @ text_top[top])

    return np.column_stack([
        audio_assigned, text_assigned,
        audio_top_sim,  text_top_sim,
        audio_margin,   text_margin,
    ]).astype(np.float32)


# 전역 정규화 임베딩 (fold마다 재계산 불필요)
_audio_norm = normalize_rows(parts["audio"])
_text_norm  = normalize_rows(parts["text"])

print("prototype features: 6 dims")
print("  [audio_class_sim, text_class_sim, audio_top_sim, text_top_sim, audio_margin, text_margin]")

In [ ]:
# ── Per-fold feature builder ──────────────────────────────────────────────────
#
# variant A/C : baseline + prototype
#   baseline  = audio(512) + text(512) + class_onehot(23) + top_onehot(5) + meta_numeric(4)
#
# variant B   : (baseline without meta) + prototype
#   = audio(512) + text(512) + class_onehot(23) + top_onehot(5) + prototype(6)

def build_x_with_proto(variant, parts_, proto_feats):
    baseline = np.hstack([
        parts_["audio"], parts_["text"],
        parts_["class"], parts_["top_class"],
        parts_["meta"],
    ])
    if variant in ("A", "C"):
        return np.hstack([baseline, proto_feats])
    elif variant == "B":
        no_meta = np.hstack([
            parts_["audio"], parts_["text"],
            parts_["class"], parts_["top_class"],
        ])
        return np.hstack([no_meta, proto_feats])
    else:
        raise ValueError(variant)


def build_fold_features(fold, train_idx, valid_idx, variant):
    """
    centroid_mask = train_idx AND confidence >= 4
    → validation 샘플은 centroid에 기여하지 않음
    """
    conf           = bsd10k_df["confidence"].to_numpy()
    centroid_mask  = np.zeros(len(bsd10k_df), dtype=bool)
    centroid_mask[train_idx] = (conf[train_idx] >= 4)

    proto_feats = compute_prototype_features(
        bsd10k_df, _audio_norm, _text_norm,
        class_categories, top_class_categories,
        centroid_mask,
    )
    x_all = build_x_with_proto(variant, parts, proto_feats)
    return x_all[train_idx].astype(np.float32), x_all[valid_idx].astype(np.float32)


# Feature dim 확인
_full_mask    = (y_conf >= 4)
_proto_preview = compute_prototype_features(
    bsd10k_df, _audio_norm, _text_norm,
    class_categories, top_class_categories,
    _full_mask,
)
for variant_ in ("A", "B", "C"):
    dim = build_x_with_proto(variant_, parts, _proto_preview).shape[1]
    print(f"v5-{variant_} feature dim: {dim}")

In [ ]:
# ── Model architectures ───────────────────────────────────────────────────────

class BinaryMLP(nn.Module):
    """v5-A, v5-B: binary 분류기 (conf >= 4 여부)"""
    def __init__(self, input_dim, hidden=(512, 256), dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden[0]),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden[0], hidden[1]),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden[1], 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class OrdinalMLP(nn.Module):
    """
    v5-C: ordinal regression 분류기
    출력: 4개 logit = [P(>=2), P(>=3), P(>=4), P(>=5)]
    filtering score = sigmoid(logit[:, 2]) = P(conf >= 4)

    왜 ordinal?
      confidence 1~5는 순서 정보가 있다.
      binary label (conf>=4 → 1)은 conf=3 vs 4 경계의 ordinal 정보를 버린다.
      ordinal target은 4개 임계값을 동시에 학습하여 이 정보를 보존한다.
    """
    def __init__(self, input_dim, hidden=(512, 256), dropout=0.3):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, hidden[0]),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden[0], hidden[1]),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Linear(hidden[1], 4)

    def forward(self, x):
        return self.head(self.backbone(x))    # (batch, 4) logits


print("BinaryMLP and OrdinalMLP defined.")

In [ ]:
# ── Dataset and training utilities ────────────────────────────────────────────

class ConfDataset(Dataset):
    """
    ordinal=True  : y는 confidence 1~5 정수 → 4개 binary threshold target으로 변환
    ordinal=False : y는 binary (0 or 1)
    """
    def __init__(self, x, y=None, ordinal=False):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = None
        if y is not None:
            if ordinal:
                targets = np.zeros((len(y), 4), dtype=np.float32)
                for k, thr in enumerate([2, 3, 4, 5]):
                    targets[:, k] = (y >= thr).astype(np.float32)
                self.y = torch.tensor(targets, dtype=torch.float32)
            else:
                self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        item = {"x": self.x[idx]}
        if self.y is not None:
            item["y"] = self.y[idx]
        return item


def predict_prob(model, loader, ordinal=False):
    """모델 추론 → filtering score (P(conf>=4)) 반환"""
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(DEVICE)
            if ordinal:
                prob = torch.sigmoid(model(x))[:, 2]   # P(>=4)
            else:
                prob = torch.sigmoid(model(x))
            preds.append(prob.detach().cpu().numpy())
    return np.concatenate(preds).astype(np.float32)


def train_fold(fold, x_train, x_valid, y_train_conf, y_valid_conf,
               model_cls, input_dim, ordinal=False):
    """
    x_train / x_valid : 이미 StandardScaler 없이 raw feature
    y_train_conf      : confidence 1~5 (ordinal target 생성에 사용)
    y_valid_conf      : 동일
    """
    seed_everything(SEED + fold)

    scaler  = StandardScaler()
    x_tr    = scaler.fit_transform(x_train).astype(np.float32)
    x_va    = scaler.transform(x_valid).astype(np.float32)

    y_bin_valid = (y_valid_conf >= 4).astype(np.int64)

    # 학습 target
    if ordinal:
        y_tr = y_train_conf          # 1~5 정수, ConfDataset이 4-head 변환
    else:
        y_tr = (y_train_conf >= 4).astype(np.int64)

    train_loader = DataLoader(
        ConfDataset(x_tr, y_tr, ordinal=ordinal),
        batch_size=RUN_CONFIG["batch_size"], shuffle=True,
    )
    valid_loader = DataLoader(
        ConfDataset(x_va), batch_size=RUN_CONFIG["batch_size"],
    )

    model = model_cls(
        input_dim,
        hidden=tuple(RUN_CONFIG["hidden"]),
        dropout=RUN_CONFIG["dropout"],
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=RUN_CONFIG["learning_rate"],
        weight_decay=RUN_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=RUN_CONFIG["epochs"]
    )
    criterion = nn.BCEWithLogitsLoss()

    best_f1, best_epoch, best_state, stale = -1.0, 0, None, 0
    history = []

    for epoch in range(RUN_CONFIG["epochs"]):
        model.train()
        total_loss, n_seen = 0.0, 0
        for batch in train_loader:
            xb = batch["x"].to(DEVICE)
            yb = batch["y"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            n_seen     += xb.size(0)
        scheduler.step()

        val_prob   = predict_prob(model, valid_loader, ordinal=ordinal)
        best_row_e, _ = pick_f1_optimal(y_bin_valid, val_prob)
        val_f1     = float(best_row_e["f1"])
        val_auc_pr = safe_auc_pr(y_bin_valid, val_prob)
        history.append({
            "epoch":     epoch,
            "train_loss": total_loss / max(n_seen, 1),
            "val_f1":     val_f1,
            "val_auc_pr": val_auc_pr,
        })

        if val_f1 > best_f1 + 1e-6:
            best_f1, best_epoch, stale = val_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= RUN_CONFIG["patience"]:
                break

    model.load_state_dict(best_state)
    val_prob  = predict_prob(model, valid_loader, ordinal=ordinal)
    best_row_f, _ = pick_f1_optimal(y_bin_valid, val_prob)
    fold_metrics = {
        "fold":       fold,
        "best_epoch": best_epoch,
        "f1":         float(best_row_f["f1"]),
        "precision":  float(best_row_f["precision"]),
        "recall":     float(best_row_f["recall"]),
        "threshold":  float(best_row_f["threshold"]),
        "auc_pr":     safe_auc_pr(y_bin_valid, val_prob),
    }
    return val_prob, fold_metrics, best_state, scaler, history


print("training utilities loaded.")

---
## 실험 v5-A
**Feature**: baseline (audio+text+class+top+meta) + prototype distance 6d  
**Target**: binary (conf >= 4)  
**질문**: prototype distance feature가 v3 binary MLP 성능을 올리는가?

In [ ]:
VARIANT_A = "A"

splitter = StratifiedKFold(n_splits=RUN_CONFIG["folds"], shuffle=True, random_state=SEED)
splits   = list(splitter.split(np.zeros(len(y_binary)), y_binary))

oof_prob_A    = np.zeros(len(y_binary), dtype=np.float32)
fold_results_A = []
artifacts_A   = []
input_dim_A   = None

print("=" * 60)
print("v5-A  |  baseline + prototype  |  binary target")
print("=" * 60)

t0 = time.time()
for fold, (train_idx, valid_idx) in enumerate(splits):
    x_tr, x_va = build_fold_features(fold, train_idx, valid_idx, VARIANT_A)
    if input_dim_A is None:
        input_dim_A = x_tr.shape[1]
        print(f"  feature dim: {input_dim_A}")

    val_prob, metrics, state, scaler, history = train_fold(
        fold, x_tr, x_va,
        y_conf[train_idx], y_conf[valid_idx],
        BinaryMLP, input_dim_A, ordinal=False,
    )
    oof_prob_A[valid_idx] = val_prob
    fold_results_A.append(metrics)
    artifacts_A.append({"state": state, "scaler": scaler})
    pd.DataFrame(history).to_csv(CKPT_DIR / f"v5A_fold{fold}_history.csv", index=False)
    print(f"  fold={fold}  f1={metrics['f1']:.4f}  auc_pr={metrics['auc_pr']:.4f}  "
          f"precision={metrics['precision']:.4f}  epoch={metrics['best_epoch']}")

print(f"elapsed: {(time.time()-t0)/60:.1f} min")

best_row_A, sweep_A = pick_f1_optimal(y_binary, oof_prob_A)
oof_A = metrics_at_threshold(y_binary, oof_prob_A, float(best_row_A["threshold"]))
print(f"\nv5-A OOF  F1={oof_A['f1']:.4f}  AUC-PR={oof_A['auc_pr']:.4f}  "
      f"precision={oof_A['precision']:.4f}  threshold={best_row_A['threshold']:.3f}")

# Save OOF
oof_A_df = bsd10k_df.copy()
oof_A_df["target_high_confidence"] = y_binary
oof_A_df["v5A_prob"] = oof_prob_A
oof_A_df.to_csv(PRED_DIR / "BSD10k_oof_v5A.csv", index=False)
print("OOF saved:", PRED_DIR / "BSD10k_oof_v5A.csv")

In [ ]:
# v5-A: fold 상세 결과
display(pd.DataFrame(fold_results_A).round(4))

---
## 실험 v5-B
**Feature**: audio + text + class + top_class + prototype 6d (**metadata numeric 제거**)  
**Target**: binary (conf >= 4)  
**질문**: title_length, tag_count, desc_length 같은 metadata numeric이 noise인가, signal인가?

In [ ]:
VARIANT_B = "B"

oof_prob_B    = np.zeros(len(y_binary), dtype=np.float32)
fold_results_B = []
artifacts_B   = []
input_dim_B   = None

print("=" * 60)
print("v5-B  |  no-metadata + prototype  |  binary target")
print("=" * 60)

t0 = time.time()
for fold, (train_idx, valid_idx) in enumerate(splits):
    x_tr, x_va = build_fold_features(fold, train_idx, valid_idx, VARIANT_B)
    if input_dim_B is None:
        input_dim_B = x_tr.shape[1]
        print(f"  feature dim: {input_dim_B}")

    val_prob, metrics, state, scaler, history = train_fold(
        fold, x_tr, x_va,
        y_conf[train_idx], y_conf[valid_idx],
        BinaryMLP, input_dim_B, ordinal=False,
    )
    oof_prob_B[valid_idx] = val_prob
    fold_results_B.append(metrics)
    artifacts_B.append({"state": state, "scaler": scaler})
    pd.DataFrame(history).to_csv(CKPT_DIR / f"v5B_fold{fold}_history.csv", index=False)
    print(f"  fold={fold}  f1={metrics['f1']:.4f}  auc_pr={metrics['auc_pr']:.4f}  "
          f"precision={metrics['precision']:.4f}  epoch={metrics['best_epoch']}")

print(f"elapsed: {(time.time()-t0)/60:.1f} min")

best_row_B, sweep_B = pick_f1_optimal(y_binary, oof_prob_B)
oof_B = metrics_at_threshold(y_binary, oof_prob_B, float(best_row_B["threshold"]))
print(f"\nv5-B OOF  F1={oof_B['f1']:.4f}  AUC-PR={oof_B['auc_pr']:.4f}  "
      f"precision={oof_B['precision']:.4f}  threshold={best_row_B['threshold']:.3f}")

oof_B_df = bsd10k_df.copy()
oof_B_df["target_high_confidence"] = y_binary
oof_B_df["v5B_prob"] = oof_prob_B
oof_B_df.to_csv(PRED_DIR / "BSD10k_oof_v5B.csv", index=False)
print("OOF saved:", PRED_DIR / "BSD10k_oof_v5B.csv")

In [ ]:
# v5-B: fold 상세 결과
display(pd.DataFrame(fold_results_B).round(4))

---
## 실험 v5-C
**Feature**: baseline + prototype 6d (v5-A와 동일)  
**Target**: ordinal — 4개 binary threshold `[P(>=2), P(>=3), P(>=4), P(>=5)]`  
**Filtering score**: `P(>=4) = sigmoid(logit[:, 2])`  

**왜 ordinal이 더 좋을 수 있는가?**
- binary target은 `conf=3 → 0`, `conf=4 → 1`로 처리 → 경계 정보 손실
- ordinal target은 `conf=3 → [1,1,0,0]`, `conf=4 → [1,1,1,0]` → conf 3과 4를 구분하는 head를 공유
- confidence 1~5의 순서 정보를 모두 활용

In [ ]:
VARIANT_C = "C"

oof_prob_C    = np.zeros(len(y_binary), dtype=np.float32)
fold_results_C = []
artifacts_C   = []
input_dim_C   = None

print("=" * 60)
print("v5-C  |  baseline + prototype  |  ORDINAL target")
print("  ordinal heads: [P(>=2), P(>=3), P(>=4), P(>=5)]")
print("  filter score : head[2] = P(conf >= 4)")
print("=" * 60)

t0 = time.time()
for fold, (train_idx, valid_idx) in enumerate(splits):
    x_tr, x_va = build_fold_features(fold, train_idx, valid_idx, VARIANT_C)
    if input_dim_C is None:
        input_dim_C = x_tr.shape[1]
        print(f"  feature dim: {input_dim_C}")

    val_prob, metrics, state, scaler, history = train_fold(
        fold, x_tr, x_va,
        y_conf[train_idx], y_conf[valid_idx],
        OrdinalMLP, input_dim_C, ordinal=True,
    )
    oof_prob_C[valid_idx] = val_prob
    fold_results_C.append(metrics)
    artifacts_C.append({"state": state, "scaler": scaler})
    pd.DataFrame(history).to_csv(CKPT_DIR / f"v5C_fold{fold}_history.csv", index=False)
    print(f"  fold={fold}  f1={metrics['f1']:.4f}  auc_pr={metrics['auc_pr']:.4f}  "
          f"precision={metrics['precision']:.4f}  epoch={metrics['best_epoch']}")

print(f"elapsed: {(time.time()-t0)/60:.1f} min")

best_row_C, sweep_C = pick_f1_optimal(y_binary, oof_prob_C)
oof_C = metrics_at_threshold(y_binary, oof_prob_C, float(best_row_C["threshold"]))
print(f"\nv5-C OOF  F1={oof_C['f1']:.4f}  AUC-PR={oof_C['auc_pr']:.4f}  "
      f"precision={oof_C['precision']:.4f}  threshold={best_row_C['threshold']:.3f}")

oof_C_df = bsd10k_df.copy()
oof_C_df["target_high_confidence"] = y_binary
oof_C_df["v5C_prob"] = oof_prob_C
oof_C_df.to_csv(PRED_DIR / "BSD10k_oof_v5C.csv", index=False)
print("OOF saved:", PRED_DIR / "BSD10k_oof_v5C.csv")

In [ ]:
# v5-C: fold 상세 결과
display(pd.DataFrame(fold_results_C).round(4))

---
## 결과 비교: v3 / v4 / v5-A / v5-B / v5-C

In [ ]:
comparison_rows = []

# v3 baseline
v3_path = ROOT / "outputs" / "confidence_filter_v3" / "predictions" / "BSD10k_oof_binary_mlp.csv"
if v3_path.exists():
    v3_df    = pd.read_csv(v3_path)
    v3_score = v3_df["predicted_high_confidence_prob"].to_numpy(dtype=float)
    v3_y     = v3_df["target_high_confidence"].to_numpy(dtype=int)
    v3_best, _ = pick_f1_optimal(v3_y, v3_score)
    comparison_rows.append({"model": "v3 Binary MLP",
                             **metrics_at_threshold(v3_y, v3_score, float(v3_best["threshold"]))})
else:
    print("v3 OOF not found; skipping.")

# v4 best (rank average binary + expected score)
v4_path = ROOT / "outputs" / "confidence_filter_v4" / "predictions" / "BSD10k_oof_v4_scores.csv"
if v4_path.exists():
    v4_df = pd.read_csv(v4_path)
    if "rank_average_binary_score" in v4_df.columns:
        v4_score = v4_df["rank_average_binary_score"].to_numpy(dtype=float)
        v4_y     = v4_df["target_high_confidence"].to_numpy(dtype=int)
        v4_best, _ = pick_f1_optimal(v4_y, v4_score)
        comparison_rows.append({"model": "v4 Rank avg (binary+score)",
                                 **metrics_at_threshold(v4_y, v4_score, float(v4_best["threshold"]))})
else:
    print("v4 OOF not found; skipping.")

# v5-A, B, C
for label, oof_prob in [
    ("v5-A  baseline+proto  binary",  oof_prob_A),
    ("v5-B  no-meta+proto   binary",  oof_prob_B),
    ("v5-C  baseline+proto  ordinal", oof_prob_C),
]:
    best_r, _ = pick_f1_optimal(y_binary, oof_prob)
    comparison_rows.append({"model": label,
                             **metrics_at_threshold(y_binary, oof_prob, float(best_r["threshold"]))})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(REPORT_DIR / "v5_comparison.csv", index=False)
display(comparison_df[["model", "f1", "auc_pr", "precision", "recall", "threshold"]].round(4))

In [ ]:
# PR curve 비교
if HAS_MATPLOTLIB:
    curve_candidates = {
        "v5-A (baseline+proto, binary)": oof_prob_A,
        "v5-B (no-meta+proto,  binary)": oof_prob_B,
        "v5-C (baseline+proto, ordinal)": oof_prob_C,
    }
    if v3_path.exists():
        curve_candidates["v3 Binary MLP"] = v3_score
    if v4_path.exists() and "rank_average_binary_score" in v4_df.columns:
        curve_candidates["v4 Rank avg"] = v4_score

    plt.figure(figsize=(7, 6))
    for name, score in curve_candidates.items():
        y_ = y_binary if "v5" in name else (v3_y if "v3" in name else v4_y)
        p, r, _ = precision_recall_curve(y_, score)
        ap = safe_auc_pr(y_, score)
        plt.plot(r, p, label=f"{name}  AP={ap:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("v5 PR curves — BSD10k OOF")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "v5_pr_curves.png", dpi=160)
    plt.show()
else:
    print("matplotlib not installed; skipping PR curve plot.")

---
## BSD35k-CS 예측 적용

In [ ]:
# BSD35k-CS 데이터 로드
print("Loading BSD35k-CS...")
bsd35k_parts = v2.load_bsd35k_parts(class_categories, top_class_categories, parts)
bsd35k_df    = bsd35k_parts["df"].copy()
print(f"BSD35k-CS samples: {len(bsd35k_df):,}")

# BSD35k-CS 임베딩 정규화
_b35_audio_norm = normalize_rows(bsd35k_parts["audio"])
_b35_text_norm  = normalize_rows(bsd35k_parts["text"])

# Prototype centroid: BSD10k의 conf>=4 샘플 전체로 계산 (fold 제약 없음)
_full_hc_mask = (y_conf >= 4)

_audio_ctr_full = build_centroids(_audio_norm, bsd10k_df["class"].astype(str).to_numpy(),
                                   class_categories,     _full_hc_mask)
_text_ctr_full  = build_centroids(_text_norm,  bsd10k_df["class"].astype(str).to_numpy(),
                                   class_categories,     _full_hc_mask)
_audio_top_full = build_centroids(_audio_norm, bsd10k_df["class_top"].astype(str).to_numpy(),
                                   top_class_categories, _full_hc_mask)
_text_top_full  = build_centroids(_text_norm,  bsd10k_df["class_top"].astype(str).to_numpy(),
                                   top_class_categories, _full_hc_mask)

print("Centroids built from BSD10k conf>=4 samples.")

In [ ]:
def compute_bsd35k_proto(target_df, target_audio_norm, target_text_norm,
                          audio_ctr, text_ctr, audio_top, text_top):
    """BSD35k-CS용 prototype feature 계산 (BSD10k centroid를 BSD35k-CS 샘플에 적용)"""
    cls_arr    = target_df["class"].astype(str).to_numpy()
    top_arr    = target_df["class_top"].astype(str).to_numpy()
    n          = len(target_df)
    cls_to_idx = {c: i for i, c in enumerate(class_categories)}

    A = np.vstack([audio_ctr[c] for c in class_categories])
    T = np.vstack([text_ctr[c]  for c in class_categories])

    all_audio_sims = target_audio_norm @ A.T
    all_text_sims  = target_text_norm  @ T.T

    audio_assigned = np.zeros(n, dtype=np.float32)
    text_assigned  = np.zeros(n, dtype=np.float32)
    audio_top_sim  = np.zeros(n, dtype=np.float32)
    text_top_sim   = np.zeros(n, dtype=np.float32)
    audio_margin   = np.zeros(n, dtype=np.float32)
    text_margin    = np.zeros(n, dtype=np.float32)

    for i in range(n):
        cls = cls_arr[i]
        top = top_arr[i]
        j   = cls_to_idx.get(cls, 0)

        audio_assigned[i] = all_audio_sims[i, j]
        text_assigned[i]  = all_text_sims[i, j]

        others_a = np.delete(all_audio_sims[i], j)
        others_t = np.delete(all_text_sims[i],  j)
        audio_margin[i] = audio_assigned[i] - (others_a.max() if len(others_a) > 0 else 0.0)
        text_margin[i]  = text_assigned[i]  - (others_t.max() if len(others_t) > 0 else 0.0)

        audio_top_sim[i] = float(target_audio_norm[i] @ audio_top[top])
        text_top_sim[i]  = float(target_text_norm[i]  @ text_top[top])

    return np.column_stack([
        audio_assigned, text_assigned,
        audio_top_sim,  text_top_sim,
        audio_margin,   text_margin,
    ]).astype(np.float32)


print("Computing BSD35k-CS prototype features...")
bsd35k_proto = compute_bsd35k_proto(
    bsd35k_df,
    _b35_audio_norm, _b35_text_norm,
    _audio_ctr_full, _text_ctr_full,
    _audio_top_full, _text_top_full,
)
print(f"done. shape: {bsd35k_proto.shape}")

In [ ]:
def build_bsd35k_x(variant):
    baseline = np.hstack([
        bsd35k_parts["audio"], bsd35k_parts["text"],
        bsd35k_parts["class"], bsd35k_parts["top_class"],
        bsd35k_parts["meta"],
    ])
    if variant in ("A", "C"):
        return np.hstack([baseline, bsd35k_proto])
    elif variant == "B":
        no_meta = np.hstack([
            bsd35k_parts["audio"], bsd35k_parts["text"],
            bsd35k_parts["class"], bsd35k_parts["top_class"],
        ])
        return np.hstack([no_meta, bsd35k_proto])


def predict_bsd35k(variant, artifacts, model_cls, input_dim, ordinal=False):
    x_raw      = build_bsd35k_x(variant).astype(np.float32)
    fold_probs = []
    for art in artifacts:
        x_scaled = art["scaler"].transform(x_raw).astype(np.float32)
        model    = model_cls(input_dim, hidden=tuple(RUN_CONFIG["hidden"]),
                             dropout=RUN_CONFIG["dropout"]).to(DEVICE)
        model.load_state_dict(art["state"])
        loader   = DataLoader(ConfDataset(x_scaled), batch_size=RUN_CONFIG["batch_size"])
        fold_probs.append(predict_prob(model, loader, ordinal=ordinal))
    return np.vstack(fold_probs).mean(axis=0).astype(np.float32)


print("Predicting BSD35k-CS...")
pred_A = predict_bsd35k("A", artifacts_A, BinaryMLP,  input_dim_A, ordinal=False)
pred_B = predict_bsd35k("B", artifacts_B, BinaryMLP,  input_dim_B, ordinal=False)
pred_C = predict_bsd35k("C", artifacts_C, OrdinalMLP, input_dim_C, ordinal=True)

thresh_A = float(best_row_A["threshold"])
thresh_B = float(best_row_B["threshold"])
thresh_C = float(best_row_C["threshold"])

out_df = bsd35k_df.copy()
out_df["v5A_prob"] = pred_A
out_df["v5B_prob"] = pred_B
out_df["v5C_prob"] = pred_C
out_df["v5A_pred"] = (pred_A >= thresh_A).astype(int)
out_df["v5B_pred"] = (pred_B >= thresh_B).astype(int)
out_df["v5C_pred"] = (pred_C >= thresh_C).astype(int)
out_df.to_csv(PRED_DIR / "BSD35k-CS_filter_predictions_v5.csv", index=False)

for label, pred, thr in [("A", pred_A, thresh_A), ("B", pred_B, thresh_B), ("C", pred_C, thresh_C)]:
    kept = int((pred >= thr).sum())
    print(f"v5-{label}  threshold={thr:.3f}  retained={kept:,}/{len(out_df):,}  "
          f"({kept/len(out_df):.3f})")

print("saved:", PRED_DIR / "BSD35k-CS_filter_predictions_v5.csv")

---
## Threshold Scenario 분석

In [ ]:
# Best v5 모델 선택
best_variant, best_oof_prob, best_pred_35k, best_thresh = max(
    [
        ("A", oof_prob_A, pred_A, thresh_A),
        ("B", oof_prob_B, pred_B, thresh_B),
        ("C", oof_prob_C, pred_C, thresh_C),
    ],
    key=lambda t: metrics_at_threshold(y_binary, t[1], t[2])["f1"],
)
print(f"Best v5 model: v5-{best_variant}")

scenario_rows = []
for thr in sorted(set([0.18, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, round(best_thresh, 3)])):
    kept  = int((best_pred_35k >= thr).sum())
    oof_m = metrics_at_threshold(y_binary, best_oof_prob, thr)
    scenario_rows.append({
        "threshold":       thr,
        "retained":        kept,
        "retained_ratio":  round(kept / len(best_pred_35k), 4),
        "oof_precision":   round(oof_m["precision"], 4),
        "oof_recall":      round(oof_m["recall"],    4),
        "oof_f1":          round(oof_m["f1"],        4),
    })

scenario_df = pd.DataFrame(scenario_rows)
scenario_df.to_csv(REPORT_DIR / "v5_bsd35k_scenarios.csv", index=False)
display(scenario_df)

In [ ]:
# 최종 리포트 저장
report_lines = [
    "# Confidence Filtering v5 보고서",
    "",
    "## 핵심 아이디어",
    "",
    "confidence = label accuracy → audio/text embedding이 assigned class와 얼마나 잘 맞는가를 직접 feature화",
    "",
    "## 신규 Feature: Class Prototype Distance (6d)",
    "",
    "| feature | 의미 |",
    "|---------|------|",
    "| audio_class_sim | audio emb ↔ assigned class 오디오 centroid cosine sim |",
    "| text_class_sim | text emb ↔ assigned class 텍스트 centroid cosine sim |",
    "| audio_top_sim | audio emb ↔ top-class 오디오 centroid cosine sim |",
    "| text_top_sim | text emb ↔ top-class 텍스트 centroid cosine sim |",
    "| audio_margin | audio_class_sim - best_other_class_sim |",
    "| text_margin | text_class_sim - best_other_class_sim |",
    "",
    "centroid는 학습 fold conf>=4 샘플만 사용 (data leakage 없음)",
    "",
    "## 실험 결과 비교",
    "",
    markdown_table(comparison_df[["model","f1","auc_pr","precision","recall","threshold"]].round(4)),
    "",
    f"## BSD35k-CS Filtering Scenarios (best: v5-{best_variant})",
    "",
    markdown_table(scenario_df),
    "",
    "## 저장 파일",
    "",
    f"- BSD10k OOF: outputs/confidence_filter_v5/predictions/BSD10k_oof_v5A.csv (B, C)",
    f"- BSD35k-CS:  outputs/confidence_filter_v5/predictions/BSD35k-CS_filter_predictions_v5.csv",
    f"- 비교표:     outputs/confidence_filter_v5/reports/v5_comparison.csv",
]

report_text = "\n".join(report_lines)
(REPORT_DIR / "confidence_filter_v5_report_ko.md").write_text(report_text, encoding="utf-8")
(ROOT / "confidence_filter_v5_report_ko.md").write_text(report_text, encoding="utf-8")

print("Report saved.")
print("\n=" * 30)
print("All v5 experiments complete.")
print("="*60)
print(comparison_df[["model","f1","auc_pr","precision"]].round(4).to_string(index=False))